# Teste: bcb_sgs.get(codigo)

Testa a fonte **BCB SGS** (Banco Central — Sistema Gerenciador de Séries Temporais).

- **API:** https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados  
- **Parâmetros:** dataInicial e dataFinal (DD/MM/YYYY), intervalo máximo 10 anos.  
- **Lógica:** Uma requisição a `ultimos/1` obtém a data do último valor; em seguida, requisições de 10 em 10 anos até não haver mais dados, montando um único JSON em ordem cronológica.

In [ ]:
import os
import sys
if os.path.basename(os.getcwd()) == "tests":
    os.chdir("..")
raiz = os.path.abspath(".")
src = os.path.join(raiz, "src")
if os.path.isdir(src) and src not in sys.path:
    sys.path.insert(0, src)
for key in list(sys.modules.keys()):
    if key == "data_economist" or key.startswith("data_economist."):
        del sys.modules[key]
from data_economist import bcb_sgs
print("OK — bcb_sgs importado.")

## 1) Série 433 (IPCA)

Código 433 = IPCA. A função obtém toda a série em janelas de 10 anos e concatena.

In [ ]:
dados = bcb_sgs.get(433)
print(f"Total de pontos: {len(dados)}")
if dados:
    print(f"Primeiro: {dados[0]}")
    print(f"Último: {dados[-1]}")
dados[:5]

## 2) Código como string

bcb_sgs.get() aceita int ou str.

In [ ]:
dados_str = bcb_sgs.get("433")
print(f"get(433) e get('433') têm o mesmo tamanho: {len(dados_str) == len(dados)}")
dados_str[-3:]

## 3) Tratamento de erro (código inexistente)

Código que não existe deve levantar exceção (HTTPError, ValueError ou Timeout).

In [ ]:
import requests
try:
    bcb_sgs.get(999999999, timeout=5)
    print("Inesperado: não levantou exceção.")
except (requests.HTTPError, ValueError, requests.exceptions.Timeout) as e:
    print(f"Esperado: {type(e).__name__}: {e}")